In [1]:
# ==============================================================================
# Fine-Tuning Pretrained DistilBERT for Cybercrime Multi-Class Classification
# Dataset: Hugging Face (Dev523/Crime-Reports-Dataset)
# Model: distilbert-base-uncased (State-of-the-Art Transformer)
# Hardware: Google Colab T4 GPU (Estimated time: ~8 to 10 minutes)
# ==============================================================================

# Step 1: Install Required Libraries
!pip install -q transformers datasets evaluate accelerate scikit-learn

import os, re, shutil, time
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import evaluate
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Using Device: {device} | GPU Active: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    print("⚠️ WARNING: GPU not detected! Please enable T4 GPU in Runtime > Change runtime type.")

# ------------------------------------------------------------------------------
# 2. Download and Clean Dataset
# ------------------------------------------------------------------------------
print("\n[Step 1/5] Downloading dataset from Hugging Face...")
url = "https://huggingface.co/datasets/Dev523/Crime-Reports-Dataset/resolve/main/train.csv"
df = pd.read_csv(url)

print(f"-> Raw Records: {len(df):,}")

# Drop missing values
df = df.dropna(subset=['crimeaditionalinfo', 'category'])

def clean_text(text: str) -> str:
    """Cleans text for Transformer Tokenizer."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' [URL] ', text) # Preserve URL token
    text = re.sub(r'[^a-zA-Z0-9\s\[\]]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

df['clean_text'] = df['crimeaditionalinfo'].apply(clean_text)
# Remove very short noisy texts (< 4 words)
df = df[df['clean_text'].apply(lambda x: len(x.split()) >= 4)]

# Filter out categories with less than 10 samples (e.g. single noise record)
category_counts = df['category'].value_counts()
valid_categories = category_counts[category_counts >= 10].index.tolist()
df = df[df['category'].isin(valid_categories)]

# ------------------------------------------------------------------------------
# 3. Smart Stratified Downsampling for Balanced Learning
# ------------------------------------------------------------------------------
print("\n[Step 2/5] Applying Smart Class Balancing (Capping majority classes)...")
# To prevent 53k 'Online Financial Fraud' from dominating, cap it to 5,000 samples
# while keeping 100% of minority class samples (Ransomware, CSAM, Cyber Terrorism, etc.)
balanced_dfs = []
for cat in valid_categories:
    cat_df = df[df['category'] == cat]
    if len(cat_df) > 5000:
        cat_df = cat_df.sample(n=5000, random_state=42)
    balanced_dfs.append(cat_df)

df_balanced = pd.concat(balanced_dfs).sample(frac=1.0, random_state=42).reset_index(drop=True)

print(f"-> Balanced Training Dataset Size: {len(df_balanced):,} records across {len(valid_categories)} classes.")
print("\nCategory Distribution after Balancing:")
print(df_balanced['category'].value_counts())

# ------------------------------------------------------------------------------
# 4. Tokenization & Dataset Preparation
# ------------------------------------------------------------------------------
print("\n[Step 3/5] Tokenizing texts with DistilBERT Tokenizer...")

# Label Mapping
unique_labels = sorted(df_balanced['category'].unique())
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for i, label in enumerate(unique_labels)}
df_balanced['label'] = df_balanced['category'].map(label2id)

# Train/Test Split (85% Train, 15% Test)
train_df, test_df = train_test_split(
    df_balanced,
    test_size=0.15,
    random_state=42,
    stratify=df_balanced['label']
)

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples['clean_text'], truncation=True, max_length=128)

train_dataset = Dataset.from_pandas(train_df[['clean_text', 'label']]).map(tokenize_function, batched=True)
test_dataset = Dataset.from_pandas(test_df[['clean_text', 'label']]).map(tokenize_function, batched=True)

# ------------------------------------------------------------------------------
# 5. Model Architecture & Training Setup
# ------------------------------------------------------------------------------
print("\n[Step 4/5] Initializing DistilBERT Classifier & Training Pipeline...")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    weighted_f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "macro_f1": macro_f1, "weighted_f1": weighted_f1}

training_args = TrainingArguments(
    output_dir="./distilbert_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True, # High speed on T4 GPU
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=100,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

# Start Fine-Tuning
print("\n" + "=" * 60)
print("              STARTING DISTILBERT FINE-TUNING                ")
print("=" * 60)
start_time = time.time()
trainer.train()
training_time = (time.time() - start_time) / 60
print(f"\n[SUCCESS] Training finished in {training_time:.1f} minutes!")

# ------------------------------------------------------------------------------
# 6. Evaluation & Detailed Classification Report
# ------------------------------------------------------------------------------
print("\n[Step 5/5] Generating Detailed Performance Evaluation...")
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

print("=" * 65)
print(f"  OVERALL ACCURACY : {accuracy_score(true_labels, preds) * 100:.2f}%")
print(f"  MACRO F1-SCORE   : {f1_score(true_labels, preds, average='macro') * 100:.2f}%")
print("=" * 65)

print("\nDetailed Per-Category Classification Report:")
print(classification_report(true_labels, preds, target_names=unique_labels, digits=4))

# ------------------------------------------------------------------------------
# 7. Save and Zip Model for 1-Click Download
# ------------------------------------------------------------------------------
EXPORT_DIR = "./cybercrime_distilbert_model"
os.makedirs(EXPORT_DIR, exist_ok=True)

model.save_pretrained(EXPORT_DIR)
tokenizer.save_pretrained(EXPORT_DIR)

# Zip for easy download
shutil.make_archive("cybercrime_distilbert_model", 'zip', EXPORT_DIR)

print("\n" + "=" * 65)
print("✅ MODEL READY FOR DOWNLOAD!")
print("-> Download file: 'cybercrime_distilbert_model.zip' from the Colab Files sidebar.")
print("=" * 65)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.6 MB/s eta 0:00:00
[INFO] Using Device: cuda | GPU Active: True

[Step 1/5] Downloading dataset from Hugging Face...
-> Raw Records: 93,686

[Step 2/5] Applying Smart Class Balancing (Capping majority classes)...
-> Balanced Training Dataset Size: 28,060 records across 14 classes.

Category Distribution after Balancing:
category
Online Financial Fraud                                  5000
Online and Social Media Related Crime                   5000
Any Other Cyber Crime                                   5000
Cyber Attack/ Dependent Crimes                          3608
RapeGang Rape RGRSexually Abusive Content               2815
Sexually Obscene material                               1795
Hacking  Damage to computercomputer system etc          1661
Sexually Explicit Act                                   1513
Cryptocurrency Crime                                     476
Online Gambling  Betting                                 433


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...


[SUCCESS] Training finished in 4.1 minutes!

[Step 5/5] Generating Detailed Performance Evaluation...


  OVERALL ACCURACY : 63.86%
  MACRO F1-SCORE   : 42.98%

Detailed Per-Category Classification Report:
                                                      precision    recall  f1-score   support

                               Any Other Cyber Crime     0.5074    0.5947    0.5476       750
Child Pornography CPChild Sexual Abuse Material CSAM     0.5556    0.2727    0.3659        55
                                Cryptocurrency Crime     0.5625    0.7500    0.6429        72
                      Cyber Attack/ Dependent Crimes     1.0000    1.0000    1.0000       541
                                     Cyber Terrorism     0.0000    0.0000    0.0000        24
      Hacking  Damage to computercomputer system etc     0.4704    0.5743    0.5172       249
                            Online Cyber Trafficking     0.0000    0.0000    0.0000        26
                              Online Financial Fraud     0.7374    0.7040    0.7203       750
                            Online Gambling  Bettin

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ MODEL READY FOR DOWNLOAD!
-> Download file: 'cybercrime_distilbert_model.zip' from the Colab Files sidebar.
